# seasonal-spirals demo

Each ring is one year. The angle is the time of year. The colour is the value.
Seasonal patterns show up as colour bands that repeat at the same angular position across rings.

All examples use Wikipedia pageview data fetched live from the Wikimedia API.

In [ ]:
from seasonal_spirals import fetch_pageviews, plot_spiral

## Influenza: strong winter seasonality

Traffic peaks every December to March as flu season hits the Northern Hemisphere.
The same bright arc appears at the same angular position in every ring.
Notice 2020-21 is noticeably quiet: COVID lockdowns suppressed flu transmission that year.

In [ ]:
influenza = fetch_pageviews("Influenza", start="2018-01-01", end="2026-03-01")

plot_spiral(
    influenza,
    title="Influenza Wikipedia pageviews 2018-2026",
).show()

## Kayaking: summer seasonality

The opposite phase. Traffic peaks May to August and the bright arc lands on the other side of the clock from Influenza.
You can see this without reading any labels.

In [ ]:
kayaking = fetch_pageviews("Kayaking", start="2018-01-01", end="2026-03-01")

plot_spiral(
    kayaking,
    title="Kayaking Wikipedia pageviews 2018-2026",
    dark_mode=True,
).show()

## Game of Thrones: event-driven spikes

No smooth seasonal pattern here. Traffic spikes sharply when each series airs (April-June for most seasons) and collapses between them.
Series 7 shifted to July-August, which is visible as the bright arc rotating clockwise by about two positions in the 2017 ring.
After the finale in 2019, traffic drops to near zero and stays there.

In [ ]:
got = fetch_pageviews("Game_of_Thrones", start="2015-01-01", end="2026-03-01")

plot_spiral(
    got,
    title="Game of Thrones Wikipedia pageviews 2015-2026",
).show()

## Sourdough: a structural break

Flat and unremarkable until March 2020, when the first COVID lockdowns triggered a baking surge.
The spiral shows it clearly: one ring looks completely different from all the others, then traffic settles at a permanently higher baseline.

In [ ]:
sourdough = fetch_pageviews("Sourdough", start="2017-01-01", end="2026-03-01")

plot_spiral(
    sourdough,
    title="Sourdough Wikipedia pageviews 2017-2026",
    dark_mode=True,
).show()

## Side by side: contrasting seasonality

Placing Influenza and Kayaking next to each other makes the phase difference obvious at a glance.
This uses the matplotlib backend, which is better suited to static multi-panel layouts.
Install it with `uv add seasonal-spirals[matplotlib]` if you haven't already.

In [ ]:
import matplotlib.pyplot as plt
from seasonal_spirals import SeasonalSpiral

fig, axes = plt.subplots(1, 2, figsize=(16, 8), facecolor="#0f1117")

for data, ax, label in [
    (influenza, axes[0], "Influenza (winter peak)"),
    (kayaking, axes[1], "Kayaking (summer peak)"),
]:
    SeasonalSpiral(data).plot(ax=ax, dark_mode=True)
    ax.set_title(label, pad=15, fontsize=11, color="#e6edf3")

fig.suptitle("Contrasting seasonal patterns", fontsize=13, y=1.01, color="#e6edf3")
plt.tight_layout()
plt.show()

## Saving outputs

Interactive plots can be saved as standalone HTML files or as static images (requires `kaleido`).

## Colour scheme: hybrid linear-log scale

The default colour scheme splits the data at a **cutoff** threshold:

- **Below the cutoff** (ordinary days): linear scale, light green to dark navy
- **Above the cutoff** (extraordinary spikes): log scale, deep blue through magenta to coral red

Both halves get equal visual weight, so the cutoff controls how much colour range is spent on ordinary variation versus spikes.

The cutoff defaults to the **Tukey IQR fence** (`Q3 + 1.5 * IQR`) — the same rule used by boxplots to flag outliers. It is robust: a handful of viral-spike days have no effect on the threshold. You can override it with a known value via `cutoff`, or supply a custom rule via `cutoff_fn`.

In [ ]:
import numpy as np

# Default: auto cutoff (Tukey IQR fence)
plot_spiral(influenza, title="Influenza - default colour scheme").show()

# Override with a known value directly
plot_spiral(influenza, title="Influenza - manual cutoff", cutoff=30_000).show()

# Custom rule: top 5% of days always get spike colours
plot_spiral(influenza, title="Influenza - p95 cutoff", cutoff_fn=lambda v: np.percentile(v, 95), dark_mode=True).show()

# MAD/Hampel: very robust, good when outliers are extreme but rare
def mad_cutoff(values):
    median = np.median(values)
    mad = np.median(np.abs(values - median))
    return median + 3 * 1.4826 * mad

plot_spiral(influenza, title="Influenza - MAD cutoff", cutoff_fn=mad_cutoff, dark_mode=True).show()

In [ ]:
fig = plot_spiral(
    influenza,
    title="Influenza Wikipedia pageviews 2018-2026",
)

fig.write_html("influenza_spiral.html")  # standalone HTML
# fig.write_image("influenza_spiral.png")     # static PNG (needs kaleido: uv add kaleido)